# Sicilian NMT with NLLB-200 — zero-shot + LoRA fine-tune

All-in-one: evaluate Meta's **NLLB-200** zero-shot on our Arba-Sicula test set, then **LoRA fine-tune** it and re-evaluate (BLEU/chrF, same as the local Sockeye baseline). Sicilian = `scn_Latn`.

**Data** is read from your Google Drive at `MyDrive/SicilianNMT-colab/data/` (the six `train/valid/test.{scn,en}` files). If they aren't there, the cell falls back to a manual upload. Artifacts (adapter, translations, scores) are saved back to Drive, which syncs to your PC.

**Steps:** Runtime → **GPU** (required). Run top to bottom.

In [17]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate

In [18]:
# Mount Drive (for data in, artifacts out).
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/SicilianNMT-colab'
except Exception:
    OUT = 'sicilian-nllb-out'
os.makedirs(OUT, exist_ok=True)
print('using', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
using /content/drive/MyDrive/SicilianNMT-colab


In [19]:
# Read data from Drive (MyDrive/SicilianNMT-colab/data/). No upload widget needed.
def read(p): return open(p, encoding='utf-8').read().splitlines()
DATA = f'{OUT}/data'
need = [f'{s}.{l}' for s in ('train', 'valid', 'test') for l in ('scn', 'en')]
if all(os.path.exists(f'{DATA}/{f}') for f in need):
    base = DATA
    print('reading data from Drive:', DATA)
else:
    print('Data not found in Drive at', DATA)
    print('Put the 6 files there (synced from your PC), or upload them now:')
    from google.colab import files
    files.upload()
    base = '.'
splits = {s: {l: read(f'{base}/{s}.{l}') for l in ('scn', 'en')} for s in ('train', 'valid', 'test')}
print({s: len(d['scn']) for s, d in splits.items()})

Data not found in Drive at /content/drive/MyDrive/SicilianNMT-colab/data
Put the 6 files there (synced from your PC), or upload them now:


Saving itsc.scn to itsc (1).scn
Saving test.en to test (1).en
Saving test.scn to test (1).scn
Saving test.tsv to test (1).tsv
Saving train.en to train (1).en
Saving train.scn to train (1).scn
Saving train.tsv to train (1).tsv
Saving valid.en to valid (1).en
Saving valid.scn to valid (1).scn
Saving valid.tsv to valid (1).tsv
{'train': 22456, 'valid': 1000, 'test': 1000}


In [20]:
import json, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu.metrics import BLEU, CHRF

MODEL = 'facebook/nllb-200-distilled-600M'   # or facebook/nllb-200-1.3B
LANG = {'scn': 'scn_Latn', 'en': 'eng_Latn'}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('WARNING: NO GPU. Runtime -> Change runtime type -> GPU, then Restart and run all.')
print('downloading/loading', MODEL, '(~2.4 GB the first time)...')
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device).eval()
results = {}

def translate(texts, src, tgt, bs=32, max_len=160):
    tok.src_lang = LANG[src]
    tgt_id = tok.convert_tokens_to_ids(LANG[tgt])
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tgt_id, max_length=max_len, num_beams=5)
        out += tok.batch_decode(gen, skip_special_tokens=True)
        print(f'  translated {min(i+bs, len(texts))}/{len(texts)}', end='\r')
    print()
    return out

def report(hyp, ref, tag):
    b = BLEU().corpus_score(hyp, [ref]).score
    c = CHRF().corpus_score(hyp, [ref]).score
    results[tag] = {'bleu': round(b, 2), 'chrf': round(c, 2)}
    fname = tag.replace(' ', '_').replace('->', '2')
    open(f'{OUT}/hyp_{fname}.txt', 'w', encoding='utf-8').write('\n'.join(hyp) + '\n')
    json.dump(results, open(f'{OUT}/results.json', 'w'), indent=2, ensure_ascii=False)
    print(f'{tag}:  BLEU={b:.2f}  chrF={c:.2f}   (saved to {OUT})')
print('ready on', device)

downloading/loading facebook/nllb-200-distilled-600M (~2.4 GB the first time)...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

ready on cuda


In [21]:
# ---- ZERO-SHOT ----  (a few minutes on GPU; progress prints below)
report(translate(splits['test']['scn'], 'scn', 'en'), splits['test']['en'],  'zero-shot scn->en')
report(translate(splits['test']['en'],  'en', 'scn'), splits['test']['scn'], 'zero-shot en->scn')

  translated 1000/1000
zero-shot scn->en:  BLEU=25.58  chrF=52.57   (saved to /content/drive/MyDrive/SicilianNMT-colab)
  translated 1000/1000
zero-shot en->scn:  BLEU=9.44  chrF=40.40   (saved to /content/drive/MyDrive/SicilianNMT-colab)


## LoRA fine-tune (scn→en)

Parameter-efficient: base weights frozen, small adapters trained on our data. Change `DIRECTION` to `en2scn` to fine-tune the other way.

In [23]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

DIRECTION, EPOCHS = 'scn2en', 3
src, tgt = DIRECTION.split('2')
tok.src_lang, tok.tgt_lang = LANG[src], LANG[tgt]

ft = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                                      target_modules=['q_proj', 'v_proj'], task_type='SEQ_2_SEQ_LM'))
ft.print_trainable_parameters()

def tokenize(b):
    return tok(b['src'], text_target=b['tgt'], max_length=160, truncation=True)
train_ds = Dataset.from_dict({'src': splits['train'][src], 'tgt': splits['train'][tgt]}).map(
    tokenize, batched=True, remove_columns=['src', 'tgt'])

args = Seq2SeqTrainingArguments(output_dir=f'{OUT}/trainer-{DIRECTION}', num_train_epochs=EPOCHS,
    per_device_train_batch_size=16, learning_rate=2e-4, fp16=(device=='cuda'),
    logging_steps=50, save_strategy='no', report_to=[])
Seq2SeqTrainer(model=ft, args=args, train_dataset=train_ds,
               data_collator=DataCollatorForSeq2Seq(tok, model=ft)).train()

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
# ---- EVALUATE FINE-TUNED + SAVE ADAPTER ----
model = ft.eval()
report(translate(splits['test'][src], src, tgt), splits['test'][tgt], f'LoRA {DIRECTION}')
ft.save_pretrained(f'{OUT}/nllb-lora-{DIRECTION}')
print('saved LoRA adapter + results.json + hyps to', OUT)
print(json.dumps(results, indent=2, ensure_ascii=False))

Everything is under `MyDrive/SicilianNMT-colab/` (adapter, `hyp_*.txt`, `results.json`) — it persists and syncs to your PC.

**Note:** NLLB's seed Sicilian uses a post-2017 orthography differing from our literary standard, so zero-shot may be penalised; fine-tuning fixes it. NLLB numbers are raw text (compare to the raw floor 5.27); Sockeye 7.24 was tokenized space.